# **First Homework Assignment**

In [2]:
import numpy as np
import pandas as pd
import re

# Used for setting display settings to increase readability.
from IPython.core.display import display_html

ModuleNotFoundError: No module named 'numpy'

In [2]:
# Custom formater.
def fromat_nan(val):
    if pd.isna(val):
        return ""
    if isinstance(val, float):
        return f"{round(val, 3)}"
    return f"{val}"

# Helper function that displays each dataframe in its own column.
def display_list(dfs, index=True, axis=-1):
    # Convert each split to HTML with proper styling for side-by-side display.
    html_str = "<table><tr>" 
    for df in dfs:
        # Dataframe styling options.
        styled_df = df.style.format(fromat_nan)
        if axis >= 0:
            styled_df = styled_df.highlight_min(axis=axis, color='darkgreen')
            styled_df = styled_df.highlight_max(axis=axis, color='darkred')
        styled_html = styled_df.to_html(index=index)

        # Enables newlines and bold titles.
        name_html = df.attrs['name'].replace('\n', '<br>')
        name_html = f"<div style='text-align: left; font-weight: bold;'>{name_html}</div>"
        html_str += f"<td style='vertical-align: top; padding: 5px;'>{name_html + styled_html}</td>"
    html_str += "</tr></table>"
    display_html(html_str, raw=True)

# Helper function to display one data frame as multiple columns.
def display(df, cols, index=True):
    # Split the dataframe into columns.
    ratio = int(np.ceil(len(df) / cols))
    df_split = [df.iloc[i * ratio: (i + 1) * ratio] for i in range(cols)]
    display_list(df_split, index)

In [3]:
# Calculates means based on axis and returns the dataframe.
def get_means(dfs, axis):
    lst = []
    for df in dfs:
        mean = df.mean(axis).to_frame(name='Mean')
        mean.attrs['name'] = df.attrs['name']
        lst.append(mean)
    return lst

## **Sequential vs Parallelized**

In [4]:
# Specifiying the parameters of the analysis.
num_cores = ["seq", 1, 2, 4, 8, 16]
repetitions = 5

_csv = pd.read_csv("results_parallel.csv")
_csv.columns = [str(c).strip().strip('"') for c in _csv.columns]
_csv["Time_sec"] = pd.to_numeric(
    _csv["Time"].astype(str).str.strip().str.replace(r"\s*s$", "", regex=True),
    errors="coerce",
)
_means = _csv.groupby(["Image", "Method", "Threads"])["Time_sec"].mean()
infiles = sorted(_csv["Image"].unique())

In [5]:
# Parsing metrics equal for all iterations.
df = pd.DataFrame(columns=num_cores, index=infiles)
df.attrs['name'] = f"Time averages over 5 repetitions"
for N in num_cores:
    method = "seam-carving" if N == "seq" else "seam-carving-omp"
    threads = 1 if N == "seq" else int(N)
    for filename in infiles:
        key = (filename, method, threads)
        df.loc[filename, N] = _means[key] if key in _means.index else np.nan

    # Miss ratio in percents.
    df.loc["Mean"] = df.mean(axis=0)

print("Maximal values per row are displayed in red, while minimal are displayed in green.")
display_list([df], axis=1)

Maximal values per row are displayed in red, while minimal are displayed in green.


,seq,1,2,4,8,16
1024x768.png,1.627,2.975,1.551,0.833,0.626,0.628
1920x1200.png,4.65,8.796,4.499,2.362,1.513,1.224
3840x2160.png,21.556,36.28,18.509,9.594,5.533,3.732
720x480.png,0.673,1.284,0.68,0.379,0.311,0.348
7680x4320.png,92.397,156.381,78.988,40.655,22.315,13.553
test.png,0.488,0.925,0.507,0.291,0.267,
valve.png,0.521,0.988,0.535,0.3,0.268,0.318
Mean,17.416,29.661,15.038,7.773,4.405,3.3


In [6]:
# Average speed-up for 16 cores: sequential time / parallel time, averaged over images.
_img_rows = [i for i in df.index if i != "Mean"]
_speedup_16 = df.loc[_img_rows, "seq"] / df.loc[_img_rows, 16]
avg_speedup_16 = _speedup_16.mean(skipna=True)
print(f"Average speed-up (16 cores vs seq.): {avg_speedup_16:.3f}x")

_speedup_7680 = df.loc["7680x4320.png", "seq"] / df.loc["7680x4320.png", 16]
print(f"Speed-up (16 cores vs seq.) for 7680x4320.png: {_speedup_7680:.3f}x")

Average speed-up (16 cores vs seq.): 3.759x
Speed-up (16 cores vs seq.) for 7680x4320.png: 6.817x


## **Dynamic Thread Allocation**

In [ ]:
def best_time_per_image(csv_df):
    g = csv_df.groupby(["Image", "E_const", "DP_const", "R_const"], as_index=False)["Time_sec"].mean()
    pick = g.loc[g.groupby("Image")["Time_sec"].idxmin()].reset_index(drop=True)
    pick = pick.rename(
        columns={
            "Image": "image",
            "Time_sec": "time",
            "E_const": "C1",
            "DP_const": "C2",
            "R_const": "C3",
        }
    )
    return pick[
        [
            "image",
            "time",
            "C1",
            "C2",
            "C3",
        ]
    ].sort_values("image")

In [8]:
# Specifiying the parameters of the analysis.
repetitions = 1

_csv_opt = pd.read_csv("results_optimized-32.csv")
_csv_opt.columns = [str(c).strip().strip('"') for c in _csv_opt.columns]
_csv_opt["Time_sec"] = pd.to_numeric(
    _csv_opt["Time"].astype(str).str.strip().str.replace(r"\s*s$", "", regex=True),
    errors="coerce",
)
infiles_opt = sorted(_csv_opt["Image"].unique())

In [ ]:
# Parsing metrics equal for all iterations.
df_opt = best_time_per_image(_csv_opt)
df_opt.attrs["name"] = f"Best heuristic per image on 16 threads"

display_list([df_opt.set_index("image")], index=True)

,time,C1,C2,C3
image,,,,
1024x768.png,0.471,500,1000,100
1920x1200.png,0.778,1000,500,100
3840x2160.png,2.502,500,1000,100
720x480.png,0.333,1000,500,100
7680x4320.png,9.154,500,1000,100
test.png,0.246,500,5000,5000
valve.png,0.271,5000,1000,500


In [23]:
# Specifiying the parameters of the analysis.
repetitions = 1

def parse_csv_optimized(name):
    _csv_opt = pd.read_csv(name)
    _csv_opt.columns = [str(c).strip().strip('"') for c in _csv_opt.columns]
    _csv_opt["Time_sec"] = pd.to_numeric(
        _csv_opt["Time"].astype(str).str.strip().str.replace(r"\s*s$", "", regex=True),
        errors="coerce",
    )
    return _csv_opt

In [20]:
df_opt32 = best_time_per_image(parse_csv_optimized("results_optimized-32.csv"))
df_opt32.attrs["name"] = f"Best heuristic per image on 32 threads"
df_opt32.set_index("image")

df_opt161 = best_time_per_image(parse_csv_optimized("results_optimized-161.csv"))
df_opt161.attrs["name"] = f"Best heuristic per image on 16 threads 1"
df_opt161.set_index("image")

df_opt162 = best_time_per_image(parse_csv_optimized("results_optimized-162.csv"))
df_opt162.attrs["name"] = f"Best heuristic per image on 16 threads 2"
df_opt162.set_index("image")

df_opt163 = best_time_per_image(parse_csv_optimized("results_optimized-163.csv"))
df_opt163.attrs["name"] = f"Best heuristic per image on 16 threads 3"
df_opt163.set_index("image")

df_opt16_new = best_time_per_image(parse_csv_optimized("results_optimized-16-new.csv"))
df_opt16_new.attrs["name"] = f"Best heuristic per image on 16 threads with adjusted constants"
df_opt16_new.set_index("image")

df_opt16_1tweak = best_time_per_image(parse_csv_optimized("results_optimized-16-1tweak.csv"))
df_opt16_1tweak.attrs["name"] = f"Best heuristic per image on 16 threads with adjusted constants"
df_opt16_1tweak.set_index("image")

display_list([df_opt32], index=True)
display_list([df_opt161, df_opt162, df_opt163], index=True)
display_list([df_opt16_new, df_opt16_1tweak], index=True)

,image,time,C1,C2,C3
0,1024x768.png,0.471,500,1000,100
1,1920x1200.png,0.778,1000,500,100
2,3840x2160.png,2.502,500,1000,100
3,720x480.png,0.333,1000,500,100
4,7680x4320.png,9.154,500,1000,100
5,test.png,0.246,500,5000,5000
6,valve.png,0.271,5000,1000,500


,image,time,C1,C2,C3
0,1024x768.png,0.412,5000,1000,100
1,1920x1200.png,0.894,5000,500,100
2,3840x2160.png,3.477,5000,1000,100
3,720x480.png,0.252,1000,1000,100
4,7680x4320.png,14.008,5000,1000,500
5,valve.png,0.212,5000,1000,100
,image,time,C1,C2,C3
0,1024x768.png,0.436,100,5000,100
1,1920x1200.png,0.907,100,500,100
2,3840x2160.png,3.219,100,1000,100


,image,time,C1,C2,C3
0,1024x768.png,0.391,1,1000,1
1,1920x1200.png,0.982,1,10000,1
2,3840x2160.png,3.518,1,1000,1
3,720x480.png,0.187,1,10000,1
4,7680x4320.png,13.439,1,1000,1
5,test.png,0.335,1,5000,1
6,valve.png,0.151,1,700,1
,image,time,C1,C2,C3
0,1024x768.png,0.353,10000,1000,1
1,1920x1200.png,0.906,10000,1000,1


In [28]:
# Mean best time per image across the four 16-thread heuristic tables (excludes 32-thread).
_dfs16 = [df_opt161, df_opt162, df_opt163, df_opt16_new]
_t = pd.concat(
    [df[["image", "time"]].assign(_run=i) for i, df in enumerate(_dfs16)],
    ignore_index=True,
)
_t["time"] = pd.to_numeric(_t["time"], errors="coerce")
mean_time_16_by_image = _t.groupby("image", sort=True)["time"].mean()
df_mean16 = mean_time_16_by_image.to_frame(name="mean time (s)")
df_mean16.attrs["name"] = "Mean best time per image: average over 4 runs"
display_list([df_mean16], index=True, axis=-1)

,mean time (s)
image,
1024x768.png,0.414
1920x1200.png,0.912
3840x2160.png,3.358
720x480.png,0.239
7680x4320.png,13.459
test.png,0.357
valve.png,0.197


In [30]:
# Mean thread counts per phase from the heuristics.
from pathlib import Path

_MAX_THR = 16


def _png_dims(path):
    import struct

    with open(path, "rb") as f:
        hdr = f.read(24)
    if len(hdr) < 24 or hdr[:8] != b"\x89PNG\r\n\x1a\n" or hdr[12:16] != b"IHDR":
        return None
    w, h = struct.unpack(">II", hdr[16:24])
    return int(w), int(h)


def _dims_for_image(name):
    m = re.match(r"^(\d+)x(\d+)\.png$", str(name), re.I)
    if m:
        return int(m.group(1)), int(m.group(2))
    for base in (Path("../src/test_images"), Path("test_images")):
        p = base / name
        if p.is_file():
            wh = _png_dims(p)
            if wh:
                return wh
    return None


def _thr_energy(max_t, w, h, c1):
    c1 = int(c1)
    threads = (w * h) // c1
    if threads > max_t:
        threads = max_t
    if threads < 1:
        threads = 1
    return threads


def _thr_dp(max_t, w, c2):
    c2 = int(c2)
    threads = w // c2
    if threads < 1:
        threads = 1
    if threads > max_t // 2:
        threads = max_t // 2
    if threads < 1:
        threads = 1
    return threads


def _thr_removal(max_t, h, c3):
    c3 = int(c3)
    threads = h // c3
    if threads > max_t:
        threads = max_t
    if threads < 1:
        threads = 1
    return threads


_rows = []
for run_i, df in enumerate(_dfs16):
    for _, r in df.iterrows():
        wh = _dims_for_image(r["image"])
        if wh is None:
            continue
        w, h = wh
        _rows.append(
            {
                "image": r["image"],
                "_run": run_i,
                "energy": _thr_energy(_MAX_THR, w, h, r["C1"]),
                "DP": _thr_dp(_MAX_THR, w, r["C2"]),
                "removal": _thr_removal(_MAX_THR, h, r["C3"]),
            }
        )

_thr_long = pd.DataFrame(_rows)
_mean_thr = _thr_long.groupby("image", sort=True)[["energy", "DP", "removal"]].mean()
_mean_thr["mean of 3 phases"] = _mean_thr.mean(axis=1)
_mean_thr.columns = ["mean energy thr.", "mean DP thr.", "mean removal thr.", "mean (3 phases)"]
_mean_thr.attrs["name"] = (
    f"Mean thread counts from heuristics (max {_MAX_THR}, DP/removal cap {_MAX_THR//2}) — avg over 4 runs"
)
display_list([_mean_thr], index=True, axis=-1)

,mean energy thr.,mean DP thr.,mean removal thr.,mean (3 phases)
image,,,,
1024x768.png,16.0,1.0,9.25,8.75
1920x1200.png,16.0,2.5,13.0,10.5
3840x2160.png,16.0,3.0,16.0,11.667
720x480.png,16.0,1.0,7.0,8.0
7680x4320.png,16.0,7.5,14.0,12.5
test.png,16.0,1.0,10.0,9.0
valve.png,16.0,1.0,7.0,8.0
